# Phase 01 — Data Collection
### Symbols: NIFTY50 · BANKNIFTY · NIFTYNEXT50

| Source | Timeframe | Period |
|--------|-----------|--------|
| Yahoo Finance (yfinance) | Daily (1d) | Jan 2000 → Mar 2026 |
| Zerodha Kite API | 1-minute | Whatever is available |

**Output** (Google Drive): `MyDrive/Regime_Detection_Swagato_M25AI2107/Data/`
```
Data/
  raw/
    daily/
      NIFTY50_daily.csv
      BANKNIFTY_daily.csv
      NIFTYNEXT50_daily.csv
    1min/
      NIFTY50_1min.csv
      BANKNIFTY_1min.csv
      NIFTYNEXT50_1min.csv
    logs/
      collection_log.csv
```
Only CSVs files.


## [DRIVE] Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted.')

Mounted at /content/drive
✅ Drive mounted.


## [LIBS] Install & Import

In [ ]:
!pip install -q yfinance kiteconnect pandas_market_calendars

import os, time, logging
from datetime import datetime, date, timedelta
import pandas as pd
import yfinance as yf
import pandas_market_calendars as mcal

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')
print('✅ Libraries ready.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.5/771.5 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.0/131.0 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.3/213.3 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.1.0 which is incompatible.
✅ Libraries read

## [CFG] Configuration — edit only here

In [ ]:
# Paths
GDRIVE_BASE = '/content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107'
DATA_ROOT   = os.path.join(GDRIVE_BASE, 'Data/raw')
DAILY_DIR   = os.path.join(DATA_ROOT, 'daily')
MIN1_DIR    = os.path.join(DATA_ROOT, '1min')
LOGS_DIR    = os.path.join(DATA_ROOT, 'logs')

for d in [DAILY_DIR, MIN1_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# Symbols : Yahoo Finance tickers for NSE indices

YFINANCE_TICKERS = {
    'NIFTY50':     '^NSEI',
    'BANKNIFTY':   '^NSEBANK',
    'NIFTYNEXT50': '^NSMIDCP',   # NIFTY Next 50 / Midcap proxy on Yahoo
}

# Zerodha Kite instrument tokens (NSE index tokens)
KITE_INSTRUMENTS = {
    'NIFTY50':     {'exchange': 'NSE', 'tradingsymbol': 'NIFTY 50',     'instrument_token': 256265},
    'BANKNIFTY':   {'exchange': 'NSE', 'tradingsymbol': 'NIFTY BANK',   'instrument_token': 260105},
    'NIFTYNEXT50': {'exchange': 'NSE', 'tradingsymbol': 'NIFTY NEXT 50','instrument_token': 270857},
}

# Date ranges
DAILY_START = '2000-01-01'
DAILY_END   = '2026-03-31'

# Kite credentials https://developers.kite.trade (need to run the login cell to get access_token each session)
KITE_API_KEY    = 'uciyfcxac7bs7ngb'
KITE_API_SECRET = '0mqk9xsvdvgb4i6gkjebng7hfehp98po'

print('✅ [CFG] loaded.')
print(f'   Daily data : {DAILY_START} → {DAILY_END}')
print(f'   Symbols    : {list(YFINANCE_TICKERS.keys())}')
print(f'   Data root  : {DATA_ROOT}')

✅ [CFG] loaded.
   Daily data : 2000-01-01 → 2026-03-31
   Symbols    : ['NIFTY50', 'BANKNIFTY', 'NIFTYNEXT50']
   Data root  : /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw


## [YFINANCE] Download Daily Data (2000–Mar 2026)

In [ ]:
yf_log = []

for symbol, ticker in YFINANCE_TICKERS.items():
    print(f'\n⬇️  Downloading {symbol} ({ticker}) ...')
    try:
        df = yf.download(
            ticker,
            start=DAILY_START,
            end=DAILY_END,
            interval='1d',
            auto_adjust=True,
            progress=False,
        )

        if df.empty:
            print(f'   ⚠️  No data returned for {symbol}')
            yf_log.append({'symbol': symbol, 'status': 'empty', 'rows': 0})
            continue

        # Flatten multi-level columns if present
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.index.name = 'date'
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
        df.columns = ['open', 'high', 'low', 'close', 'volume']
        df = df.sort_index()

        out_path = os.path.join(DAILY_DIR, f'{symbol}_daily.csv')
        df.to_csv(out_path)

        print(f'   ✅ {len(df)} rows | {df.index[0].date()} → {df.index[-1].date()}')
        print(f'   💾 Saved: {out_path}')
        yf_log.append({
            'symbol': symbol, 'status': 'ok', 'rows': len(df),
            'start': str(df.index[0].date()), 'end': str(df.index[-1].date()),
        })

    except Exception as e:
        print(f'   ❌ Error: {e}')
        yf_log.append({'symbol': symbol, 'status': f'error: {e}', 'rows': 0})

print('\n── Yahoo Finance Summary ──')
print(pd.DataFrame(yf_log).to_string(index=False))


⬇️  Downloading NIFTY50 (^NSEI) ...
   ✅ 4546 rows | 2007-09-17 → 2026-03-30
   💾 Saved: /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/daily/NIFTY50_daily.csv

⬇️  Downloading BANKNIFTY (^NSEBANK) ...
   ✅ 4561 rows | 2007-09-17 → 2026-03-30
   💾 Saved: /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/daily/BANKNIFTY_daily.csv

⬇️  Downloading NIFTYNEXT50 (^NSMIDCP) ...
   ✅ 4252 rows | 2007-09-17 → 2026-03-30
   💾 Saved: /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/daily/NIFTYNEXT50_daily.csv

── Yahoo Finance Summary ──
     symbol status  rows      start        end
    NIFTY50     ok  4546 2007-09-17 2026-03-30
  BANKNIFTY     ok  4561 2007-09-17 2026-03-30
NIFTYNEXT50     ok  4252 2007-09-17 2026-03-30


## [KITE-LOGIN] Zerodha Kite Authentication

1. `KITE_API_KEY` and `KITE_API_SECRET` will be needed in [CFG]
2. Run this cell → a login URL will be printed
3. Open the URL, log in, copy the `request_token` from the redirect URL
4. Paste it below and run the next cell

In [ ]:
from kiteconnect import KiteConnect

kite = KiteConnect(api_key=KITE_API_KEY)
login_url = kite.login_url()
print('🔗 Open this URL in your browser, log in, then copy the request_token from the redirect URL:')
print(login_url)

🔗 Open this URL in your browser, log in, then copy the request_token from the redirect URL:
https://kite.zerodha.com/connect/login?api_key=uciyfcxac7bs7ngb&v=3


In [ ]:
# Paste the request_token from the redirect URL here
# https://127.0.0.1:3000/?request_token=yrazPxu6kT3AQh8j8gCuvFu40JTxdDUY&action=login&type=login&status=success
REQUEST_TOKEN = 'yrazPxu6kT3AQh8j8gCuvFu40JTxdDUY'   # ← fill after login

data = kite.generate_session(REQUEST_TOKEN, api_secret=KITE_API_SECRET)
kite.set_access_token(data['access_token'])
print('✅ Kite authenticated. Access token set.')

✅ Kite authenticated. Access token set.


## [GAP-FILL] Fill Missing Daily Bars from Kite

This cell:
1. Loads each daily CSV saved by [YFINANCE]
2. Finds all NSE trading days in the date range using `pandas_market_calendars`
3. Compares against what yfinance returned — identifies missing dates
4. For any missing date fetches 1-day bars from Kite
5. Merges the filled rows back into the CSV (yfinance rows always win on overlap)



In [ ]:
# NSE trading calendar
nse_cal  = mcal.get_calendar('NSE')
schedule = nse_cal.schedule(start_date=DAILY_START, end_date=DAILY_END)
all_trading_days = set(schedule.index.date)

gap_fill_log = []

for symbol, info in KITE_INSTRUMENTS.items():
    csv_path = os.path.join(DAILY_DIR, f'{symbol}_daily.csv')

    if not os.path.exists(csv_path):
        print(f'SKIP {symbol}: CSV not found.')
        continue

    # Load existing daily data
    df = pd.read_csv(csv_path, index_col='date', parse_dates=True)
    df.index = pd.to_datetime(df.index).normalize()
    existing_days = set(df.index.date)

    # Missing NSE trading days
    missing_days = sorted(all_trading_days - existing_days)

    print(f'\n{symbol}:')
    print(f'  Trading days expected : {len(all_trading_days)}')
    print(f'  Days in CSV           : {len(existing_days)}')
    print(f'  Missing total         : {len(missing_days)}')

    if not missing_days:
        print(f'  OK: no missing days.')
        gap_fill_log.append({
            'symbol': symbol,
            'missing_total': 0,
            'filled_from_kite': 0,
        })
        continue

    # Chunk missing days (reduce API calls)
    chunks = []
    c_start = missing_days[0]
    c_end   = missing_days[0]

    for d in missing_days[1:]:
        if (d - c_end).days <= 7:
            c_end = d
        else:
            chunks.append((c_start, c_end))
            c_start = d
            c_end   = d
    chunks.append((c_start, c_end))

    print(f'  Fetching {len(chunks)} chunk(s) from Kite (daily)...')

    filled_frames = []
    missing_set = set(missing_days)

    for (c_start, c_end) in chunks:
        try:
            records = kite.historical_data(
                instrument_token = info['instrument_token'],
                from_date        = c_start,
                to_date          = c_end,
                interval         = 'day',   # ✅ direct daily
            )

            if not records:
                print(f'    {c_start} -> {c_end}: empty')
                continue

            chunk_df = pd.DataFrame(records)
            chunk_df['date'] = pd.to_datetime(chunk_df['date'])
            chunk_df = chunk_df.set_index('date')

            # Normalize index
            chunk_df.index = chunk_df.index.normalize()

            # Keep only missing days
            chunk_df = chunk_df[chunk_df.index.map(lambda x: x.date() in missing_set)]

            if not chunk_df.empty:
                filled_frames.append(chunk_df)
                print(f'    {c_start} -> {c_end}: filled {len(chunk_df)} day(s)')
            else:
                print(f'    {c_start} -> {c_end}: no matching days')

        except Exception as e:
            print(f'    {c_start} -> {c_end}: ERROR {e}')

        time.sleep(0.35)

    if not filled_frames:
        print(f'  NOTE: Kite had no data for missing days (likely holidays mismatch)')
        gap_fill_log.append({
            'symbol': symbol,
            'missing_total': len(missing_days),
            'filled_from_kite': 0,
        })
        continue

    # Merge
    new_rows = pd.concat(filled_frames)
    new_rows.index.name = 'date'

    # Remove timezone if any
    if new_rows.index.tz is not None:
        new_rows.index = new_rows.index.tz_localize(None)

    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)

    merged = pd.concat([df, new_rows])

    # yfinance priority
    merged = merged[~merged.index.duplicated(keep='first')]
    merged = merged.sort_index()

    merged.to_csv(csv_path)

    print(f'  OK: filled {len(new_rows)} day(s). CSV updated -> {csv_path}')

    gap_fill_log.append({
        'symbol': symbol,
        'missing_total': len(missing_days),
        'filled_from_kite': len(new_rows),
    })

print('\n-- Gap-Fill Summary --')
print(pd.DataFrame(gap_fill_log).to_string(index=False))


NIFTY50:
  Trading days expected : 6494
  Days in CSV           : 4546
  Missing total         : 1954
  Fetching 32 chunk(s) from Kite (daily)...
    2000-01-03 -> 2007-09-14: ERROR interval exceeds max limit: 2000 days
    2007-11-09 -> 2007-11-09: filled 1 day(s)
    2008-10-28 -> 2008-10-28: filled 1 day(s)
    2009-01-01 -> 2009-01-01: filled 1 day(s)
    2010-11-05 -> 2010-11-05: filled 1 day(s)
    2011-05-31 -> 2011-05-31: filled 1 day(s)
    2011-07-14 -> 2011-07-14: filled 1 day(s)
    2011-10-26 -> 2011-10-26: filled 1 day(s)
    2011-11-24 -> 2011-11-24: filled 1 day(s)
    2011-12-26 -> 2012-01-02: filled 2 day(s)
    2012-05-21 -> 2012-05-21: filled 1 day(s)
    2012-08-28 -> 2012-08-28: filled 1 day(s)
    2012-10-26 -> 2012-10-26: filled 1 day(s)
    2012-11-13 -> 2012-11-13: filled 1 day(s)
    2013-01-01 -> 2013-01-01: filled 1 day(s)
    2014-01-01 -> 2014-01-01: filled 1 day(s)
    2014-02-17 -> 2014-02-17: filled 1 day(s)
    2014-10-23 -> 2014-10-23: filled 1 day(

## [KITE] Download 1-Minute Data

Kite allows fetching 1-min data **60 days per request**. This cell loops through
60-day chunks from the earliest allowed date to today and concatenates everything.

> Kite typically provides 1-min historical data for about **2 years** for indices.

In [ ]:
KITE_1MIN_CHUNK_DAYS = 60
kite_log = []

# Ensure datetime format (VERY IMPORTANT)
DAILY_START = pd.to_datetime(DAILY_START)
DAILY_END   = pd.to_datetime(DAILY_END)

def fetch_kite_1min(kite, instrument_token, symbol):
    """Fetch 1-min data cleanly with minimal logs."""

    all_chunks = []

    # Smart optimization: avoid useless old range
    approx_kite_start = DAILY_END - timedelta(days=750)
    current = max(DAILY_START, approx_kite_start)
    end_date = DAILY_END

    while current <= end_date:
        chunk_end = min(current + timedelta(days=KITE_1MIN_CHUNK_DAYS - 1), end_date)

        try:
            records = kite.historical_data(
                instrument_token = instrument_token,
                from_date        = current,
                to_date          = chunk_end,
                interval         = 'minute',
            )

            if records:
                all_chunks.append(pd.DataFrame(records))

        except Exception as e:
            print(f'   ❌ Error in chunk {current.date()} → {chunk_end.date()}: {e}')

        current = chunk_end + timedelta(days=1)
        time.sleep(0.35)

    if not all_chunks:
        return pd.DataFrame()

    df = pd.concat(all_chunks, ignore_index=True)

    # Clean formatting
    df = df.rename(columns={'date': 'datetime'})
    df['datetime'] = pd.to_datetime(df['datetime'])

    df = df.set_index('datetime').sort_index()
    df = df[['open', 'high', 'low', 'close', 'volume']]

    # Remove duplicates
    df = df[~df.index.duplicated(keep='last')]

    return df


# -------- MAIN LOOP -------- #

for symbol, info in KITE_INSTRUMENTS.items():
    print(f'\n⬇️ Fetching 1-min data for {symbol} ...')

    try:
        df = fetch_kite_1min(kite, info['instrument_token'], symbol)

        if df.empty:
            print(f'   ⚠️ No data available (likely beyond Kite range)')
            kite_log.append({'symbol': symbol, 'status': 'empty', 'rows': 0})
            continue

        # Save
        out_path = os.path.join(MIN1_DIR, f'{symbol}_1min.csv')
        df.to_csv(out_path)

        # Clean summary logs
        print(f'   ✅ Rows fetched: {len(df)}')
        print(f'   📅 Coverage: {df.index.min()} → {df.index.max()}')

        # Yearly summary (last 5 years only)
        yearly = df.resample('Y').size()
        print('   📊 Rows per year (last 5):')
        print(yearly.tail(5).to_string())

        print(f'   💾 Saved: {out_path}')

        kite_log.append({
            'symbol': symbol,
            'status': 'ok',
            'rows': len(df),
            'start': str(df.index.min()),
            'end': str(df.index.max()),
        })

    except Exception as e:
        print(f'   ❌ Error: {e}')
        kite_log.append({
            'symbol': symbol,
            'status': f'error: {e}',
            'rows': 0
        })


# -------- SUMMARY -------- #

print('\n── Kite 1-Min Summary ──')
print(pd.DataFrame(kite_log).to_string(index=False))


⬇️ Fetching 1-min data for NIFTY50 ...
   ✅ Rows fetched: 186594
   📅 Coverage: 2024-03-11 09:15:00+05:30 → 2026-03-30 15:29:00+05:30
   📊 Rows per year (last 5):
datetime
2024-12-31 00:00:00+05:30    72909
2025-12-31 00:00:00+05:30    91185
2026-12-31 00:00:00+05:30    22500
Freq: YE-DEC
   💾 Saved: /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/1min/NIFTY50_1min.csv

⬇️ Fetching 1-min data for BANKNIFTY ...


/tmp/ipykernel_1347/4228085321.py:78: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  yearly = df.resample('Y').size()


   ✅ Rows fetched: 186593
   📅 Coverage: 2024-03-11 09:15:00+05:30 → 2026-03-30 15:29:00+05:30
   📊 Rows per year (last 5):
datetime
2024-12-31 00:00:00+05:30    72909
2025-12-31 00:00:00+05:30    91184
2026-12-31 00:00:00+05:30    22500
Freq: YE-DEC
   💾 Saved: /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/1min/BANKNIFTY_1min.csv

⬇️ Fetching 1-min data for NIFTYNEXT50 ...


/tmp/ipykernel_1347/4228085321.py:78: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  yearly = df.resample('Y').size()


   ✅ Rows fetched: 186594
   📅 Coverage: 2024-03-11 09:15:00+05:30 → 2026-03-30 15:29:00+05:30
   📊 Rows per year (last 5):
datetime
2024-12-31 00:00:00+05:30    72909
2025-12-31 00:00:00+05:30    91185
2026-12-31 00:00:00+05:30    22500
Freq: YE-DEC
   💾 Saved: /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/1min/NIFTYNEXT50_1min.csv

── Kite 1-Min Summary ──
     symbol status   rows                     start                       end
    NIFTY50     ok 186594 2024-03-11 09:15:00+05:30 2026-03-30 15:29:00+05:30
  BANKNIFTY     ok 186593 2024-03-11 09:15:00+05:30 2026-03-30 15:29:00+05:30
NIFTYNEXT50     ok 186594 2024-03-11 09:15:00+05:30 2026-03-30 15:29:00+05:30


/tmp/ipykernel_1347/4228085321.py:78: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  yearly = df.resample('Y').size()


## [LOG] Save Collection Log

In [ ]:
log_rows = []

for row in yf_log:
    log_rows.append({'source': 'yfinance', 'timeframe': '1day', **row})

for row in gap_fill_log:
    log_rows.append({'source': 'kite_gap_fill', 'timeframe': '1day', **row})

for row in kite_log:
    log_rows.append({'source': 'kite', 'timeframe': '1min', **row})

log_df = pd.DataFrame(log_rows)
log_df['collected_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

log_path = os.path.join(LOGS_DIR, 'collection_log.csv')
log_df.to_csv(log_path, index=False)
print('Saved:', log_path)
print(log_df.to_string(index=False))

Saved: /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/logs/collection_log.csv
       source timeframe      symbol status     rows                     start                       end  missing_total  filled_from_kite        collected_at
     yfinance      1day     NIFTY50     ok   4546.0                2007-09-17                2026-03-30            NaN               NaN 2026-04-20 02:40:45
     yfinance      1day   BANKNIFTY     ok   4561.0                2007-09-17                2026-03-30            NaN               NaN 2026-04-20 02:40:45
     yfinance      1day NIFTYNEXT50     ok   4252.0                2007-09-17                2026-03-30            NaN               NaN 2026-04-20 02:40:45
kite_gap_fill      1day     NIFTY50    NaN      NaN                       NaN                       NaN         1954.0              27.0 2026-04-20 02:40:45
kite_gap_fill      1day   BANKNIFTY    NaN      NaN                       NaN                       NaN         1938.

## [VERIFY] Check What's on Drive

In [ ]:
import glob

print('-- Files on Google Drive --\n')
all_csvs = sorted(glob.glob(os.path.join(DATA_ROOT, '**/*.csv'), recursive=True))
for path in all_csvs:
    size_kb = os.path.getsize(path) / 1024
    rel     = os.path.relpath(path, DATA_ROOT)
    print(f'  {rel:45s}  {size_kb:8.1f} KB')
print(f'\nTotal files: {len(all_csvs)}')
print('\nPhase 01 complete.')

-- Files on Google Drive --

  1min/BANKNIFTY_1min.csv                         11297.8 KB
  1min/NIFTY50_1min.csv                           11298.5 KB
  1min/NIFTYNEXT50_1min.csv                       11297.2 KB
  daily/BANKNIFTY_daily.csv                         367.3 KB
  daily/NIFTY50_daily.csv                           332.9 KB
  daily/NIFTYNEXT50_daily.csv                       352.4 KB
  logs/collection_log.csv                             0.8 KB

Total files: 7

Phase 01 complete.


## Phase 01 Complete

| File | Description |
|------|-------------|
| `daily/NIFTY50_daily.csv` | Daily OHLCV, 2000–Mar 2026 |
| `daily/BANKNIFTY_daily.csv` | Daily OHLCV, 2000–Mar 2026 |
| `daily/NIFTYNEXT50_daily.csv` | Daily OHLCV, 2000–Mar 2026 |
| `1min/NIFTY50_1min.csv` | 1-min OHLCV, ~2 years from Kite |
| `1min/BANKNIFTY_1min.csv` | 1-min OHLCV, ~2 years from Kite |
| `1min/NIFTYNEXT50_1min.csv` | 1-min OHLCV, ~2 years from Kite |
| `logs/collection_log.csv` | Run summary |
